In [14]:
# clone DECODE repository
import os
import subprocess

if not os.path.exists("../decode"):
    subprocess.run(["git", "clone", "https://github.com/waldnerf/decode.git", "../decode"])
    

In [15]:
# download pretrained model from airbus france india dataset (Wang et al. 2022)

import urllib.request
os.makedirs("../model", exist_ok=True)
if not os.path.exists("../model/airbus_france_india.params"):
    urllib.request.urlretrieve(
        "https://zenodo.org/records/7315090/files/india_Airbus_SPOT_model.params?download=1",
        "../model/airbus_france_india.params")

In [16]:
import sys
sys.path.append('/data/Aldhani/cv_fields/code/decode/')
sys.path.append('/data/Aldhani/cv_fields/code/')
sys.path.append('/data/Aldhani/cv_fields/code/smallholder-fields/')

In [17]:
import glob
import time
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from osgeo import gdal
from osgeo import ogr

import mxnet as mx
from mxnet import nd, gpu, gluon, autograd, npx, image
from mxnet.gluon.data import DataLoader
from mxnet.gluon.data.vision import transforms
from mxnet.gluon.loss import Loss

In [18]:
from decode.FracTAL_ResUNet.nn.loss.mtsk_loss import *
from decode.FracTAL_ResUNet.models.heads.head_cmtsk import *
from decode.FracTAL_ResUNet.models.semanticsegmentation.FracTAL_ResUNet_features import *
from decode.FracTAL_ResUNet.models.semanticsegmentation.FracTAL_ResUNet import FracTAL_ResUNet_cmtsk

from helpers_model import *

In [19]:
# image and label paths
# file naming must align between images and labels
# current naming convention is {country}_tile_{epsg}_{tile_x}_{tile_y}_{yyyymmdd}.tif for images and 
# {country}_tile_{epsg}_{tile_x}_{tile_y}_{yyyymmdd}_mtsk.tif for labels
# for example: mozambique_tile_32636_-0019_-0279_20230503.tif

img_folder = '../sample_data/'
lbl_folder = '../labels/'

In [20]:
# create chips of 256x256 pixels for training
image_names = glob.glob(img_folder+'*.tif')
label_names = glob.glob(lbl_folder+'*_mtsk.tif')

print(len(label_names))
print(len(image_names))

if not os.path.exists(img_folder+'/chips'): os.makedirs(img_folder+'/chips')
if not os.path.exists(lbl_folder+'/chips'): os.makedirs(lbl_folder+'/chips')

start = 0
step = 256
n=0

for n, _ in enumerate(image_names):
  image = image_names[n]
  label = label_names[n]

  ds = gdal.Open(image_names[n])
  width = ds.RasterXSize
  height = ds.RasterYSize
  print(width)
  print(height)
  x_count = int(np.floor(width/step))
  y_count = int(np.floor(height/step))

  for x_step in range(0, x_count):
        for y_step in range(0, y_count):
            window = (x_step*step, y_step*step, step, step)
            out_image = f'{img_folder}/chips/{os.path.basename(image)[:-4]}_{x_step:02d}_{y_step:02d}.tif'
            out_label = f'{lbl_folder}/chips/{os.path.basename(label)[:-4]}_{x_step:02d}_{y_step:02d}.tif'
            if not os.path.exists(out_image):
                  gdal.Translate(out_image, image, srcWin = window, creationOptions=['COMPRESS=LZW'])
            if not os.path.exists(out_label):
                  gdal.Translate(out_label, label, srcWin = window, creationOptions=['COMPRESS=LZW'])


1
1
2048
2048


In [21]:
# adjust image and label paths to folder containing chips
img_folder = '../sample_data/chips'
lbl_folder = '../labels/chips'

In [ ]:
# delete label chips without positive label
lbl_fls = sorted(glob.glob(lbl_folder + '/*.tif'))
rm = [os.remove(lbl_fl) for lbl_fl in lbl_fls if np.nansum(gdal.Open(lbl_fl).ReadAsArray()[0,:,:]) == 0]

In [23]:
# random train val test split

# get tile ids from labels
tile_ids = sorted(glob.glob(lbl_folder + '/*.tif'))
print(len(tile_ids))

# create substrings for image and label paths,
# needs adjusting if file naming convention changes
image_names = sorted([f"{img_folder}/{os.path.basename(tile_id)[:-15]}_{os.path.basename(tile_id)[-9:]}" for tile_id in tile_ids])
label_names = sorted([f"{lbl_folder}/{os.path.basename(tile_id)}" for tile_id in tile_ids])

# print examples / length of images & labels
m = [print(image_names[i], label_names[i]) for i in range(0,3)]
print(len(tile_ids), len(image_names), len(label_names))

# define split 
probs=[0.60, 0.20, 0.20]

### randomly select w leakage across tiles
np.random.seed(42)
split = np.random.choice(3, len(tile_ids), replace=True, p=probs)
train_image_names = np.array(image_names)[split==0]
val_image_names = np.array(image_names)[split==1]
test_image_names = np.array(image_names)[split==2]

train_label_names = np.array(label_names)[split==0]
val_label_names = np.array(label_names)[split==1]
test_label_names = np.array(label_names)[split==2]

print('N train, val, test')
print(len(train_image_names), len(train_label_names), len(val_image_names), len(val_label_names), len(test_image_names), len(test_label_names))


25
../sample_data/chips/mozambique_tile_32636_00020_-0281_20230917_00_06.tif ../labels/chips/mozambique_tile_32636_00020_-0281_20230917_mtsk_00_06.tif
../sample_data/chips/mozambique_tile_32636_00020_-0281_20230917_00_07.tif ../labels/chips/mozambique_tile_32636_00020_-0281_20230917_mtsk_00_07.tif
../sample_data/chips/mozambique_tile_32636_00020_-0281_20230917_01_00.tif ../labels/chips/mozambique_tile_32636_00020_-0281_20230917_mtsk_01_00.tif
25 25 25
N train, val, test
17 17 4 4 4 4


In [24]:
##############################################
# training params
pretrained_model = '../model/airbus_france_india.params'

model_type = 'fractal-resunet'

epochs = 10
lr = 0.001 
lr_decay = None
n_filters = 32
depth = 6
n_classes = 1
batch_size = 8

month = 'spot'
bands='rgb'
train_val='60-20'
lossf='ftnmt_masked'
folder_suffix = 'finetune-v01'

ctx_name = 'gpu'
gpu_id = 0

In [26]:
# execute training run
model = run(train_image_names, val_image_names,
    train_label_names, val_label_names,
    trained_model=pretrained_model,
    epochs=epochs, lr=lr, lr_decay=lr_decay,
    model_type=model_type, n_filters=n_filters, depth=depth, n_classes=n_classes,
    batch_size=batch_size, month=month, bands=bands,
    train_val=train_val, normalize='none', maskedloss=True, otf_aug=True, lossf=lossf,
    ctx_name=ctx_name,
    gpu_id=gpu_id,
    folder_suffix=folder_suffix)

depth:= 0, nfilters: 32, nheads::8, widths::1
depth:= 1, nfilters: 64, nheads::16, widths::1
depth:= 2, nfilters: 128, nheads::32, widths::1
depth:= 3, nfilters: 256, nheads::64, widths::1
depth:= 4, nfilters: 512, nheads::128, widths::1
depth:= 5, nfilters: 1024, nheads::256, widths::1
depth:= 6, nfilters: 512, nheads::256, widths::1
depth:= 7, nfilters: 256, nheads::128, widths::1
depth:= 8, nfilters: 128, nheads::64, widths::1
depth:= 9, nfilters: 64, nheads::32, widths::1
depth:= 10, nfilters: 32, nheads::16, widths::1
maskedlossfct


Validation epoch 1: 100%|██████████| 1/1 [00:00<00:00,  5.40it/s]


Epoch 1:
    Train loss 0.628, accuracy 0.456, F1-score 0.000, MCC: 0.000, Dice: 0.000
    Val loss 0.587, accuracy 0.693, F1-score 0.000, MCC: 0.000, Dice: 0.000


Validation epoch 2: 100%|██████████| 1/1 [00:00<00:00,  5.20it/s]


Epoch 2:
    Train loss 0.607, accuracy 0.607, F1-score 0.000, MCC: 0.000, Dice: 0.000
    Val loss 0.577, accuracy 0.643, F1-score 0.000, MCC: 0.000, Dice: 0.000


...

Epoch 8:
    Train loss 0.555, accuracy 0.777, F1-score 0.000, MCC: 0.000, Dice: 0.000
    Val loss 0.542, accuracy 0.744, F1-score 0.000, MCC: 0.000, Dice: 0.000


Training epoch 9:  67%|██████▋   | 2/3 [00:01<00:00,  1.57it/s]/data/Aldhani/cv_fields/code/smallholder-fields/code/helpers_model.py:127: RuntimeWarning: invalid value encountered in double_scalars
  return 2. * np.sum(intersection) / (np.sum(x) + np.sum(y))
Validation epoch 9: 100%|██████████| 1/1 [00:00<00:00,  5.36it/s]


Epoch 9:
    Train loss 0.561, accuracy 0.784, F1-score 0.000, MCC: 0.000, Dice: nan
    Val loss 0.536, accuracy 0.773, F1-score 0.000, MCC: 0.000, Dice: 0.000


Validation epoch 10: 100%|██████████| 1/1 [00:00<00:00,  5.86it/s]

Epoch 10:
    Train loss 0.551, accuracy 0.840, F1-score 0.000, MCC: 0.000, Dice: 0.000
    Val loss 0.538, accuracy 0.748, F1-score 0.000, MCC: 0.000, Dice: 0.000
